# PACE Phase 11E — Resume AMPGAN v2 inference (one-run Colab)

**Analysis only. Do not edit the original manuscript, supplementary data or reviewer responses.**

This notebook restores the **exact verified AMPGAN v2 `v2.0` checkpoint** if your Colab session expired, then runs the previously prepared source-aware inference compatibility probe. It validates official Git LFS weights by SHA-256, and reports genuine generated sequences **only if** upstream sampling actually runs and yields new FASTA/CSV output.

**How to use:** Google Colab → **File > Upload notebook** → **Runtime > Run all**. Download **one** `PACE_Phase11E_Resume_Results.zip` at the end and upload it to the conversation. No PACE data or Phase 8 ZIP needs uploading for this source/inference step. The four original weight files are *not* included in the downloaded results.

**Important:** This is a compatibility probe, not an assurance that the historic TensorFlow model runs on modern Colab. If it fails, the archive will contain diagnostic logs. No generative model results are inferred or fabricated.


## 1 — Restore or reuse the original tagged checkpoint (SHA-256 verification)

In [ ]:
# PHASE 11D: Official AMPGAN v2 checkpoint recovery + source capture.
# This notebook intentionally DOES NOT synthesize peptides or claim a comparison.
import os,sys,json,hashlib,subprocess,shutil,time,zipfile,platform,re
from pathlib import Path

OUT = Path('/content/PACE_Phase11D_Audit')
OUT.mkdir(parents=True,exist_ok=True)
REPO = Path('/content/ampgan_v2_official')
SOURCE_URL = 'https://gitlab.com/vail-uvm/amp-gan.git'
TAG = 'v2.0'
EXPECTED_COMMIT = '2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70'
MODEL_PREFIX = 'models/amp_gan_2020-07-08_24/gan_1606/'
REQUIRED = [
 MODEL_PREFIX + 'saved_model.pb',
 MODEL_PREFIX + 'variables/variables.data-00000-of-00002',
 MODEL_PREFIX + 'variables/variables.data-00001-of-00002',
 MODEL_PREFIX + 'variables/variables.index'
]
meta={
 'phase':'11D',
 'scope':'AMPGAN v2 original checkpoint recovery and inference-source verification ONLY',
 'official_origin':SOURCE_URL,'expected_tag':TAG,'expected_commit':EXPECTED_COMMIT,
 'platform':platform.platform(), 'python':sys.version,
 'checkpoint_files':{},'success':False,'generated_sequences_count':0,
 'notes':['No peptides generated or benchmarked by this audit.'],
}

def execute(cmd, cwd=None, env=None, check=True, timeout=None):
 print('Running:', ' '.join(map(str,cmd)),flush=True)
 p=subprocess.run(cmd,cwd=str(cwd) if cwd else None,env=env,text=True,
                  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
 print((p.stdout or '')[-6500:],flush=True)
 if check and p.returncode!=0:raise RuntimeError('Command failed (%s): %s'%(p.returncode,' '.join(cmd)))
 return p

def sha256(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()

def lfs_pointer_at_tag(rel):
 raw=execute(['git','show',TAG+':'+rel],cwd=REPO).stdout.strip()
 size=re.search(r'(?m)^size (\d+)\s*$',raw)
 oid=re.search(r'(?m)^oid sha256:([a-f0-9]{64})\s*$',raw)
 if not (size and oid):raise RuntimeError('Missing official Git LFS pointer at tag: '+rel)
 return int(size.group(1)),oid.group(1)

try:
 if not shutil.which('git'): raise RuntimeError('git executable missing')
 chk=execute(['git','lfs','version'],check=False)
 if chk.returncode!=0:
  execute(['apt-get','update','-qq'])
  execute(['apt-get','install','-y','-qq','git-lfs'])
  execute(['git','lfs','install'])
 meta['git_lfs_version']=execute(['git','lfs','version']).stdout.strip()
 if (REPO/'.git').exists():
  print('Reusing existing local git checkout: ',REPO)
 else:
  env=os.environ.copy();env['GIT_LFS_SKIP_SMUDGE']='1'
  execute(['git','clone','--branch',TAG,'--depth','1',SOURCE_URL,str(REPO)],env=env)
  execute(['git','lfs','install','--local'],cwd=REPO)
 commit=execute(['git','rev-parse','HEAD'],cwd=REPO).stdout.strip()
 meta['commit']=commit
 meta['remote']=execute(['git','remote','get-url','origin'],cwd=REPO).stdout.strip()
 assert commit==EXPECTED_COMMIT, 'Release commit mismatch; stopping'
 assert 'vail-uvm/amp-gan' in meta['remote'].lower(), 'Unexpected repository origin'
 # Only four checkpoint files; avoid pulling hundreds of historical training inputs.
 pat=MODEL_PREFIX+'**'
 p=execute(['git','lfs','pull','--include',pat,'--exclude',''],cwd=REPO,check=False)
 meta['lfs_pull_returncode']=p.returncode
 meta['lfs_pull_tail']=(p.stdout or '')[-2500:]
 all_ok=True
 for rel in REQUIRED:
  expected_size,expected_hash=lfs_pointer_at_tag(rel)
  f=REPO/rel
  size=f.stat().st_size if f.exists() else None
  # Exact blob checksum validates that we have real LFS objects and correct versions.
  h=sha256(f) if size is not None else None
  valid=(size==expected_size and h==expected_hash)
  all_ok=all_ok and valid
  meta['checkpoint_files'][rel]={'actual_bytes':size,'expected_bytes':expected_size,
    'sha256_actual':h,'sha256_expected':expected_hash,'valid_lfs_blob':valid}
  print(rel,'VERIFIED' if valid else 'MISSING_OR_POINTER',flush=True)
 # Always export official Python generation source to permit faithful next-step inference.
 src=OUT/'source_files';src.mkdir(exist_ok=True)
 for rel in ['README.md','ampgan/README.md','ampgan/generate_samples.py',
             'ampgan/model.py','ampgan/cli_utils.py','ampgan/data_utils.py',
             'ampgan/train.py','.gitattributes']:
  origin=REPO/rel
  if origin.exists() and origin.stat().st_size<250000:
   f=src/rel;f.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(origin,f)
 meta['source_files_captured']=[str(x.relative_to(src)) for x in src.rglob('*') if x.is_file()]
 meta['success']=bool(all_ok)
 meta['conclusion']=('ORIGINAL MODEL FILES VERIFIED; GENERATION SCRIPT READY TO REVIEW'
     if all_ok else 'INCOMPLETE: one or more original Git LFS checkpoint files not retrieved')
except Exception as e:
 meta['error']=type(e).__name__+': '+str(e)
 meta['conclusion']='BLOCKED: inspect error and official upstream availability'
finally:
 (OUT/'checkpoint_recovery_report.json').write_text(json.dumps(meta,indent=2))
 (OUT/'README.txt').write_text('Authentic AMPGAN v2 checkpoint and source readiness audit.\n'
    'Generated peptides: NONE. No predicted MIC/HC50 comparison conducted.\n'
    'Do not mistake LFS pointer files for model weights.\n'
    'Retain Colab runtime if checkpoint recovery succeeded for the next inference step.\n')
 ZIP = Path('/content/PACE_Phase11D_AMPGANv2_Checkpoint_Readiness.zip')
 with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
  for f in OUT.rglob('*'):
   if f.is_file():z.write(f,f.relative_to(OUT))
 with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None
 print('\nFINAL STATUS:',meta.get('conclusion'),flush=True)
 print('OUTPUT FILE:',ZIP,'BYTES:',ZIP.stat().st_size,flush=True)
 print('IMPORTANT: SavedModel weights excluded from ZIP to avoid huge upload.\n',flush=True)
 print('Checkpoint diagnostic ZIP saved for final combined package.')


## 2 — Test official sampling / gather compatibility diagnostics

In [ ]:
from pathlib import Path
probe_file=Path('/content/PACE_Phase11E_AMPGANv2_Official_Inference_Probe.py')
SOURCE='"""PACE Phase 11E - official AMPGAN v2 inference compatibility and provenance-only probe.\n\nIntended to run in the SAME Google Colab runtime where Phase 11D verified the exact\nGit LFS objects. The script never changes the upstream checkout or retrains a model.\nIt only invokes the official generation script if its CLI and length/count cap can\nbe identified safely; otherwise it exports the official source and diagnostics.\n"""\nfrom __future__ import annotations\nimport ast, csv, hashlib, importlib.util, json, os, platform, re, shlex, subprocess, sys, time, zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nROOT=Path(\'/content\')\nREPO=ROOT/\'ampgan_v2_official\'\nOUT=ROOT/\'PACE_Phase11E_Official_Inference_Probe\'\nOUT.mkdir(parents=True,exist_ok=True)\nOUTPUT_ZIP=ROOT/\'PACE_Phase11E_AMPGANv2_Inference_Probe_Results.zip\'\nEXPECTED_COMMIT=\'2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70\'\nMODEL_REL=\'models/amp_gan_2020-07-08_24/gan_1606\'\nEXPECTED={\n MODEL_REL+\'/saved_model.pb\':(\'a5e7cafa16c33010bd0fe22747c89f05d74f8781deac9eeeb91c6e4c371177e2\',1344320),\n MODEL_REL+\'/variables/variables.data-00000-of-00002\':(\'df6b16408b05f21a6fdb1d675c12562257d619382d08af2fad391d7ed847084a\',53347),\n MODEL_REL+\'/variables/variables.data-00001-of-00002\':(\'012fc17624f1fe6b0441622bebc71ee968e4d7db80bb7910e22af93b49d69890\',12982248),\n MODEL_REL+\'/variables/variables.index\':(\'7936298d0a7ac1802bbedc726d29e27911c3aa60778ac97395a24d5f82456902\',12646),\n}\nSEQ_SET=set(\'ACDEFGHIKLMNPQRSTVWY\')\nCOUNT_FLAGS=(\'--num_samples\',\'--n_samples\',\'--num-samples\',\'--n-samples\',\'--num_sequences\',\'--num-sequences\',\'--num_seqs\',\'--sample_count\',\'--n\',\'-n\',\'--count\',\'--samples\')\nMODEL_FLAGS=(\'--model\',\'--model_dir\',\'--model-dir\',\'--model_path\',\'--model-path\',\'-m\')\nOUT_FLAGS=(\'--output\',\'--output_dir\',\'--output-dir\',\'--output_path\',\'--output-path\',\'--outfile\',\'--out\',\'-o\')\n\n\ndef hash_file(p):\n h=hashlib.sha256()\n with p.open(\'rb\') as fd:\n  for block in iter(lambda:fd.read(1024*1024),b\'\'):h.update(block)\n return h.hexdigest()\n\n\ndef invoke(cmd,cwd=None,secs=120):\n print(\'COMMAND\',shlex.join(map(str,cmd)),flush=True)\n try:\n  p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,\n      timeout=secs,env={**os.environ,\'TF_CPP_MIN_LOG_LEVEL\':\'2\',\'PYTHONUNBUFFERED\':\'1\'})\n  return {\'command\':list(map(str,cmd)), \'returncode\':p.returncode,\'output\':p.stdout[-50000:], \'timeout\':False}\n except subprocess.TimeoutExpired as e:\n  out=e.stdout or b\'\'\n  if isinstance(out,bytes):out=out.decode(errors=\'replace\')\n  return {\'command\':list(map(str,cmd)), \'returncode\':None,\'output\':out[-50000:], \'timeout\':True}\n except Exception as e:\n  return {\'command\':list(map(str,cmd)), \'returncode\':None,\'output\':repr(e),\'timeout\':False}\n\n\ndef snapshot(paths):\n d={}\n for base in paths:\n  if not base.exists():continue\n  for p in base.rglob(\'*\'):\n   if not p.is_file() or \'.git\' in p.parts or \'__pycache__\' in p.parts:continue\n   if p.is_relative_to(OUT) or p==OUTPUT_ZIP:continue\n   try:\n    st=p.stat()\n    d[str(p)]={\'size\':st.st_size,\'mtime_ns\':st.st_mtime_ns}\n   except OSError:pass\n return d\n\n\ndef flags_from_ast(path):\n source=path.read_text(errors=\'replace\')\n tree=ast.parse(source)\n flags=[]; all_arguments=[]\n for node in ast.walk(tree):\n  if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute) and node.func.attr==\'add_argument\':\n   argv=[]\n   for a in node.args:\n    if isinstance(a,ast.Constant) and isinstance(a.value,str):argv.append(a.value)\n   item={\'positional_or_flags\':argv,\'kwargs\':{}}\n   for kw in node.keywords:\n    if kw.arg and isinstance(kw.value,ast.Constant) and isinstance(kw.value.value,(str,int,float,bool,type(None))):\n     item[\'kwargs\'][kw.arg]=kw.value.value\n   all_arguments.append(item)\n   flags.extend([f for f in argv if f.startswith(\'-\')])\n return flags,all_arguments\n\n\ndef fasta_parse(path):\n seqs=[];buf=[]\n try:\n  with path.open(errors=\'replace\') as f:\n   first=f.readline().strip()\n   if not first.startswith(\'>\'):return []\n   for line in [first]+list(f):\n    t=line.strip()\n    if not t:continue\n    if t.startswith(\'>\'):\n     if buf:seqs.append(\'\'.join(buf).upper());buf=[]\n    else:buf.append(t)\n   if buf:seqs.append(\'\'.join(buf).upper())\n except (OSError,UnicodeError):return []\n return [s for s in seqs if 5<=len(s)<=100 and set(s)<=SEQ_SET]\n\n\ndef csv_parse(path):\n try:\n  with path.open(newline=\'\',errors=\'replace\') as f:\n   reader=csv.DictReader(f)\n   cols=reader.fieldnames or []\n   names=[c for c in cols if c and c.strip().lower() in {\'sequence\',\'seq\',\'peptide\',\'fasta\',\'generated_sequence\'}]\n   if not names:return []\n   ss=[]\n   for row in reader:\n    s=(row.get(names[0]) or \'\').strip().upper()\n    if 5<=len(s)<=100 and set(s)<=SEQ_SET:ss.append(s)\n   return ss\n except Exception:return []\n\n\ndef save_json(path,x):path.write_text(json.dumps(x,indent=2,default=str))\n\n\ndef main():\n data={\'phase\':\'11E\',\'purpose\':\'Original-tagged checkpoint + official generator compatibility test\',\n  \'official_tag\':\'v2.0\',\'expected_commit\':EXPECTED_COMMIT,\'repo\':str(REPO),\n  \'sampling_policy\':\'only execute if count flag and model flag are found in upstream argparse\',\n  \'model_origin\':\'official GitLab vail-uvm/amp-gan\',\'model_files\':{},\'model_verified\':False,\n  \'generated_sequence_count\':0,\'generation_conclusion\':\'NOT ATTEMPTED\',\'started_utc\':datetime.now(timezone.utc).isoformat()}\n try:\n  if not (REPO/\'.git\').exists():\n   data[\'generation_conclusion\']=\'REPO MISSING: reuse Phase 11D Colab runtime or rerun checkpoint recovery\'\n   return\n  rev=invoke([\'git\',\'rev-parse\',\'HEAD\'],cwd=REPO)\n  data[\'commit_actual\']=rev[\'output\'].strip()\n  if data[\'commit_actual\']!=EXPECTED_COMMIT:\n   data[\'generation_conclusion\']=\'REPO COMMIT MISMATCH; stopped\'\n   return\n  data[\'python\']=sys.version;data[\'platform\']=platform.platform()\n  for mod in [\'tensorflow\',\'keras\',\'numpy\',\'Bio\',\'pandas\',\'scipy\']:\n   data[\'imports_\'+mod]=bool(importlib.util.find_spec(mod))\n  for rel,(expected_hash,expected_size) in EXPECTED.items():\n   path=REPO/rel; sz=path.stat().st_size if path.exists() else None\n   sha=hash_file(path) if sz==expected_size else None\n   data[\'model_files\'][rel]={\'size\':sz,\'expected_size\':expected_size,\'sha256\':sha,\n    \'valid\':sz==expected_size and sha==expected_hash}\n  data[\'model_verified\']=all(z[\'valid\'] for z in data[\'model_files\'].values())\n  if not data[\'model_verified\']:\n   data[\'generation_conclusion\']=\'MODEL CHECKSUM MISMATCH; stopped\'\n   return\n  src=REPO/\'ampgan/generate_samples.py\'\n  if not src.exists():\n   data[\'generation_conclusion\']=\'OFFICIAL GENERATION SCRIPT MISSING\';return\n  (OUT/\'official_generate_samples.py\').write_bytes(src.read_bytes())\n  data[\'generation_source_sha256\']=hash_file(src)\n  readme=REPO/\'README.md\'\n  if readme.exists(): (OUT/\'official_README.md\').write_bytes(readme.read_bytes())\n  try:\n   flags,args=flags_from_ast(src)\n  except Exception as e:\n   data[\'generation_conclusion\']=\'Could not AST-parse official script: \'+repr(e)\n   return\n  data[\'official_argparse_flags\']=flags\n  save_json(OUT/\'original_cli_arguments.json\',args)\n  (OUT/\'official_generator_source_numbered.txt\').write_text(\'\\n\'.join(f\'{i:4d}: {s}\' for i,s in enumerate(src.read_text(errors=\'replace\').splitlines(),1)))\n  if not data[\'imports_tensorflow\']:\n   data[\'generation_conclusion\']=\'TensorFlow absent from Colab runtime. Installation/compatibility needs review.\'\n   return\n  help_run=invoke([sys.executable,str(src),\'--help\'],cwd=REPO,secs=120)\n  data[\'official_help_returncode\']=help_run[\'returncode\']\n  (OUT/\'official_help_log.txt\').write_text(help_run[\'output\'])\n  if help_run[\'returncode\']!=0:\n   data[\'generation_conclusion\']=\'Official CLI did not start; inspect help log for dependency/runtime mismatch\'\n   return\n  mf=next((f for f in MODEL_FLAGS if f in flags),None)\n  nf=next((f for f in COUNT_FLAGS if f in flags),None)\n  if not (mf and nf):\n   data[\'generation_conclusion\']=\'CLI requires manual review of model/count arguments before safe generation\'\n   return\n  model_dir=str(REPO/MODEL_REL)\n  cmd=[sys.executable,str(src),mf,model_dir,nf,\'200\']\n  # Do NOT invent --output path if a supported option cannot be proved.\n  data[\'official_invocation\']=cmd\n  pre=snapshot([REPO,ROOT])\n  start=time.time()\n  run=invoke(cmd,cwd=REPO,secs=900)\n  data[\'inference_returncode\']=run[\'returncode\']\n  data[\'inference_timeout\']=run[\'timeout\']\n  data[\'inference_seconds\']=round(time.time()-start,2)\n  (OUT/\'original_generation_log.txt\').write_text(run[\'output\'])\n  post=snapshot([REPO,ROOT]);changed=[]\n  for path,attr in post.items():\n   if path not in pre or pre[path]!=attr:\n    p=Path(path)\n    if p.suffix.lower() in {\'.fasta\',\'.fa\',\'.faa\',\'.csv\',\'.tsv\',\'.txt\',\'.json\',\'.npy\',\'.npz\'}:\n     changed.append({\'path\':path,**attr})\n  data[\'new_or_updated_candidate_artifacts\']=changed\n  if run[\'returncode\']!=0:\n   data[\'generation_conclusion\']=\'Official inference FAILED. No sequences accepted as generated.\'\n   return\n  candidates=[];sources=[]\n  for x in changed:\n   p=Path(x[\'path\'])\n   if p.is_relative_to(OUT) or \'train\' in p.name.lower() or x[\'size\']>10000000:continue\n   arr=fasta_parse(p) if p.suffix.lower() in {\'.fa\',\'.faa\',\'.fasta\'} else csv_parse(p) if p.suffix.lower()==\'.csv\' else []\n   if arr:\n    sources.append({\'path\':str(p),\'raw_valid_rows\':len(arr),\'sha256\':hash_file(p)})\n    candidates.extend(arr)\n  uniq=list(dict.fromkeys(candidates))\n  data[\'parsed_new_output_sources\']=sources\n  data[\'generated_sequence_count\']=len(uniq)\n  if uniq:\n   fasta=OUT/\'AMPGANv2_OFFICIAL_CANDIDATES_UNVERIFIED_BIOACTIVITY.fasta\'\n   fasta.write_text(\'\'.join(f\'>AMPGANv2_v2.0_generated_{i:04d}\\n{s}\\n\' for i,s in enumerate(uniq,1)))\n   data[\'generated_fasta_sha256\']=hash_file(fasta)\n   data[\'generation_conclusion\']=\'Official script completed; newly generated files yielded candidate peptides. Authenticity and condition settings require final audit.\'\n  else:\n   data[\'generation_conclusion\']=\'Official script succeeded, but no parseable new FASTA/CSV was discovered; original outputs must be examined.\'\n except Exception as e:\n  data[\'generation_conclusion\']=\'PROBE ERROR; inspect diagnostic: \'+type(e).__name__+\': \'+str(e)\n finally:\n  data[\'finished_utc\']=datetime.now(timezone.utc).isoformat()\n  save_json(OUT/\'phase11e_probe_status.json\',data)\n  (OUT/\'README.txt\').write_text(\'Phase 11E official AMPGAN v2 inference-provenance probe.\\n\'\n   \'No fabricated peptides; only an original release and SHA-256 verified checkpoint.\\n\'\n   \'Sequences, if present, are GENERATED PREDICTIONS, not validated MIC/hemolysis.\\n\'\n   \'Original author manuscript/data files were not edited.\\n\'\n   \'A successful checkpoint verification alone does not guarantee inference compatibility.\\n\')\n  with zipfile.ZipFile(OUTPUT_ZIP,\'w\',zipfile.ZIP_DEFLATED) as z:\n   for p in OUT.rglob(\'*\'):\n    if p.is_file():z.write(p,p.relative_to(OUT))\n  with zipfile.ZipFile(OUTPUT_ZIP) as z:assert z.testzip() is None\n  print(\'\\nPHASE 11E:\',data[\'generation_conclusion\'])\n  print(\'Generated candidate sequences:\',data[\'generated_sequence_count\'])\n  print(\'RESULT ZIP:\',OUTPUT_ZIP,\'size\',OUTPUT_ZIP.stat().st_size)\n  print(\'Keep the Colab runtime active if model files are verified.\')\n  print(\'Inference diagnostic ZIP saved for final combined package.\')\n\nif __name__==\'__main__\':main()\n'
probe_file.write_text(SOURCE)
print('Probe script compiled: ',probe_file)
exec(compile(SOURCE,str(probe_file),'exec'),{'__name__':'__main__'})


## 3 — Download one small diagnostics archive

In [ ]:
from pathlib import Path
import zipfile, json, hashlib
from google.colab import files
out=Path('/content/PACE_Phase11E_Resume_Results.zip')
name_map={
 'Phase11D_checkpoint_audit.zip': Path('/content/PACE_Phase11D_AMPGANv2_Checkpoint_Readiness.zip'),
 'Phase11E_inference_probe.zip': Path('/content/PACE_Phase11E_AMPGANv2_Inference_Probe_Results.zip')
}
assert all(p.is_file() for p in name_map.values()), 'Missing one diagnostic ZIP: inspect preceding cells'
manifest={}
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
 for arc,p in name_map.items():
  data=p.read_bytes()
  manifest[arc]={'bytes':len(data),'sha256':hashlib.sha256(data).hexdigest()}
  z.writestr(arc,data)
 z.writestr('INSTRUCTIONS.txt','Upload this ZIP to ChatGPT for Phase 11E analysis. Neither actual model weights nor the original author data are modified.\n')
 z.writestr('manifest.json',json.dumps(manifest,indent=2))
with zipfile.ZipFile(out) as z: assert z.testzip() is None
print('One-run result file:',out,'size:',out.stat().st_size,'bytes')
files.download(str(out))
